# 01 - InstructBLIP Zero-Shot Inference\n\nRuns `Salesforce/instructblip-vicuna-7b` on 650 test images with checkpoint/resume.

In [1]:
!pip -q install transformers accelerate pillow pandas scikit-learn tqdm sentencepiece

In [6]:
import os, re, ast, json
from pathlib import Path
import pandas as pd
import torch
from tqdm import tqdm
from PIL import Image
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
      print('GPU:', torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Data setup\nUpload `colab_experiments/` folder to Drive and set `BASE_DIR`.

In [15]:
# Check what's actually in your Drive
import os
from pathlib import Path

base = Path('/content/drive/MyDrive/colab_experiments')

print("=== Folder Structure ===")
print(f"\nBase exists: {base.exists()}")

if base.exists():
    print(f"\nContents of colab_experiments/:")
    for item in sorted(os.listdir(base)):
        print(f"  - {item}")

    utils_path = base / 'utils'
    print(f"\nutils/ exists: {utils_path.exists()}")

    if utils_path.exists():
        print(f"\nContents of utils/:")
        for item in sorted(os.listdir(utils_path)):
            print(f"  - {item}")

=== Folder Structure ===

Base exists: True

Contents of colab_experiments/:
  - data.zip
  - notebooks
  - results
  - utils

utils/ exists: True

Contents of utils/:
  - data_loader.py
  - metrics.py
  - prompt_templates.py


In [17]:
# Test if we can read the file directly
utils_path = Path('/content/drive/MyDrive/colab_experiments/utils/data_loader.py')
print(f"File exists: {utils_path.exists()}")
print(f"File size: {utils_path.stat().st_size} bytes")

# Try to read first few lines
with open(utils_path) as f:
    lines = f.readlines()[:5]
    print(f"\nFirst 5 lines of data_loader.py:")
    for line in lines:
        print(line.rstrip())

File exists: True
File size: 2721 bytes

First 5 lines of data_loader.py:
"""
Data loading utilities for Colab inference experiments.
"""

from __future__ import annotations


In [18]:
import sys
from pathlib import Path

BASE_DIR = Path('/content/drive/MyDrive/colab_experiments')
utils_dir = str(BASE_DIR / 'utils')

print(f"Utils directory: {utils_dir}")
print(f"Already in sys.path: {utils_dir in sys.path}")

if utils_dir not in sys.path:
    sys.path.insert(0, utils_dir)
    print(f"Added to sys.path")

print(f"\nCurrent sys.path (first 5):")
for i, p in enumerate(sys.path[:5]):
    print(f"  {i}: {p}")

# Now try importing
print("\n=== Attempting import ===")
try:
    import data_loader
    print("✓ data_loader imported successfully!")
    print(f"  Module location: {data_loader.__file__}")
except Exception as e:
    print(f"✗ Import failed: {e}")

Utils directory: /content/drive/MyDrive/colab_experiments/utils
Already in sys.path: True

Current sys.path (first 5):
  0: /content
  1: /env/python
  2: /usr/lib/python312.zip
  3: /usr/lib/python3.12
  4: /usr/lib/python3.12/lib-dynload

=== Attempting import ===
✗ Import failed: No module named 'data_loader'


In [20]:
import sys
from pathlib import Path

BASE_DIR = Path('/content/drive/MyDrive/colab_experiments')
utils_dir = str(BASE_DIR / 'utils')

# Remove it if it exists anywhere
while utils_dir in sys.path:
    sys.path.remove(utils_dir)

# Force it to FIRST position
sys.path.insert(0, utils_dir)

print(f"sys.path[0]: {sys.path[0]}")
print(f"Expected: {utils_dir}")
print(f"Match: {sys.path[0] == utils_dir}")

# Now try importing
print("\n=== Attempting import ===")
import data_loader
print("✓ SUCCESS!")
print(f"Module: {data_loader.__file__}")

# Test the function
from data_loader import load_metadata
print("✓ load_metadata imported!")

sys.path[0]: /content/drive/MyDrive/colab_experiments/utils
Expected: /content/drive/MyDrive/colab_experiments/utils
Match: True

=== Attempting import ===


ModuleNotFoundError: No module named 'data_loader'

In [21]:
from pathlib import Path

utils_dir = Path('/content/drive/MyDrive/colab_experiments/utils')
data_loader_file = utils_dir / 'data_loader.py'

print("=== Testing file syntax ===")
try:
    with open(data_loader_file) as f:
        code = f.read()

    # Try to compile it
    compile(code, str(data_loader_file), 'exec')
    print("✓ File compiles without syntax errors")
    print(f"File length: {len(code)} characters")

    # Try to execute it in a namespace
    namespace = {}
    exec(code, namespace)
    print(f"✓ File executes successfully")
    print(f"Defined names: {[k for k in namespace.keys() if not k.startswith('_')]}")

except SyntaxError as e:
    print(f"✗ Syntax error: {e}")
except Exception as e:
    print(f"✗ Execution error: {e}")

=== Testing file syntax ===
✓ File compiles without syntax errors
File length: 2721 characters
✓ File executes successfully
Defined names: ['annotations', 'ast', 'Path', 'pd', 'Image', 'parse_label', 'load_metadata', 'load_image', 'InferenceDataset']


In [22]:
from google.colab import drive
drive.mount('/content/drive')

# Paths
from pathlib import Path
BASE_DIR = Path('/content/drive/MyDrive/colab_experiments')
DATA_DIR = BASE_DIR / 'data'
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load utilities by direct execution (bypass import system)
print("Loading utilities...")

# Load data_loader.py
with open(BASE_DIR / 'utils/data_loader.py') as f:
    exec(f.read(), globals())

# Load prompt_templates.py
with open(BASE_DIR / 'utils/prompt_templates.py') as f:
    exec(f.read(), globals())

# Load metrics.py
with open(BASE_DIR / 'utils/metrics.py') as f:
    exec(f.read(), globals())

print("✓ Utilities loaded successfully")

# Test it works
metadata_df = load_metadata(DATA_DIR / 'metadata.csv')
print(f'✓ Loaded {len(metadata_df)} rows')
metadata_df.head(2)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading utilities...
✓ Utilities loaded successfully


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/colab_experiments/data/metadata.csv'

In [23]:
import zipfile
import os

zip_path = BASE_DIR / 'data.zip'
extract_to = BASE_DIR / 'data'

print(f"ZIP file exists: {zip_path.exists()}")
print(f"ZIP file size: {zip_path.stat().st_size / 1e6:.1f} MB")

if not extract_to.exists():
    print(f"\nExtracting to: {extract_to}")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(BASE_DIR)
    print("✓ Extraction complete!")
else:
    print(f"✓ Data folder already exists")

# Verify extraction
print(f"\nContents of data/:")
for item in sorted(os.listdir(extract_to))[:10]:
    print(f"  - {item}")

# Check image count
image_dir = extract_to / 'test_images'
if image_dir.exists():
    image_count = len(list(image_dir.glob('*.jpg')))
    print(f"\n✓ Found {image_count} images in test_images/")

ZIP file exists: True
ZIP file size: 370.4 MB

Extracting to: /content/drive/MyDrive/colab_experiments/data
✓ Extraction complete!

Contents of data/:
  - explanations.csv
  - generate_explanations_template.py
  - metadata.csv
  - prepare_data.py
  - test_images

✓ Found 650 images in test_images/


In [24]:
# Load and verify metadata
metadata_df = load_metadata(DATA_DIR / 'metadata.csv')
print(f'✓ Loaded {len(metadata_df)} rows')
print(f'✓ Columns: {list(metadata_df.columns)}')
metadata_df.head(2)


✓ Loaded 650 rows
✓ Columns: ['image_id', 'category', 'ocr_text', 'human_label', 'image_path', 'explanation_implicit', 'human_label_vec']


,image_id,category,ocr_text,human_label,image_path,explanation_implicit,human_label_vec
0,TE-104.jpg,BRAND_DEPENDENT,इतनी tastyBiryani खाने के बादभी HEAL MY FEELINGS,"[1,0,0,0,0,0,0]",test_images/TE-104.jpg,,"[1, 0, 0, 0, 0, 0, 0]"
1,TE-177.jpg,BRAND_DEPENDENT,Porn hub 0 Previous 102 ^ 103| 101!* /05N 105|...,"[1,0,0,0,1,0,0]",test_images/TE-177.jpg,,"[1, 0, 0, 0, 1, 0, 0]"


In [25]:
model_id = 'Salesforce/instructblip-vicuna-7b'

print("Loading InstructBLIP model...")
print("This will download ~14GB (first time only)")
print("Expected time: 5-10 minutes\n")

from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration

processor = InstructBlipProcessor.from_pretrained(model_id)
model = InstructBlipForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,  # Use FP16 on A100
    device_map='auto'
)
model.eval()

print("\n✓ Model loaded successfully!")
print(f"✓ Device: {next(model.parameters()).device}")

Loading InstructBLIP model...
This will download ~14GB (first time only)
Expected time: 5-10 minutes



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]


✓ Model loaded successfully!
✓ Device: cuda:0


In [26]:
import re

def _extract_pred(text: str):
    m = re.search(r'PREDICTION:\s*\[([01,\s]+)\]', text, flags=re.IGNORECASE)
    if not m:
        return None
    arr = [x.strip() for x in m.group(1).split(',')]
    if len(arr) == 7 and all(x in ('0','1') for x in arr):
        return '[' + ','.join(arr) + ']'
    return None

final_csv = RESULTS_DIR / 'instructblip_final.csv'
checkpoint_interval = 50

done = {}
if final_csv.exists():
    prev = pd.read_csv(final_csv)
    done = {r['image_id']: r for _, r in prev.iterrows()}
    print(f'Resuming from {len(done)} rows')

rows = []
print(f"Starting inference on {len(metadata_df)} images...")
print(f"Checkpoints every {checkpoint_interval} images")
print(f"Estimated time: ~3-4 hours on A100\n")

for idx, row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):
    iid = row['image_id']
    if iid in done:
        rows.append(done[iid])
        continue
    try:
        image = Image.open(DATA_DIR / row['image_path']).convert('RGB')
        prompt = format_instructblip_prompt()
        inputs = processor(images=image, text=prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=128)
        text = processor.batch_decode(out, skip_special_tokens=True)[0]
        pred = _extract_pred(text)
        rec = {
            'image_id': iid,
            'category': row['category'],
            'prediction_raw': text,
            'prediction_vec': pred if pred else '[0,0,0,0,0,0,0]',
            'parse_ok': bool(pred),
            'human_label': row['human_label']
        }
    except Exception as e:
        rec = {
            'image_id': iid,
            'category': row['category'],
            'prediction_raw': f'ERROR: {e}',
            'prediction_vec': '[0,0,0,0,0,0,0]',
            'parse_ok': False,
            'human_label': row['human_label']
        }
    rows.append(rec)
    if len(rows) % checkpoint_interval == 0:
        pd.DataFrame(rows).to_csv(final_csv, index=False)
        print(f'✓ Checkpoint saved: {len(rows)} images')

res_df = pd.DataFrame(rows)
res_df.to_csv(final_csv, index=False)
print(f'\n🎉 Inference complete!')
print(f'✓ Saved: {final_csv}')
print(f'✓ Total images: {len(res_df)}')
print(f'✓ Parse success rate: {res_df["parse_ok"].mean():.1%}')

Starting inference on 650 images...
Checkpoints every 50 images
Estimated time: ~3-4 hours on A100



  8%|▊         | 50/650 [00:25<04:55,  2.03it/s]

✓ Checkpoint saved: 50 images


 15%|█▌        | 100/650 [00:49<04:56,  1.86it/s]

✓ Checkpoint saved: 100 images


...

✓ Checkpoint saved: 550 images


 92%|█████████▏| 600/650 [04:42<00:23,  2.17it/s]

✓ Checkpoint saved: 600 images


100%|██████████| 650/650 [05:06<00:00,  2.12it/s]

✓ Checkpoint saved: 650 images

🎉 Inference complete!
✓ Saved: /content/drive/MyDrive/colab_experiments/results/instructblip_final.csv
✓ Total images: 650
✓ Parse success rate: 0.0%


In [27]:
# Load results and inspect predictions
res_df = pd.read_csv(final_csv)

print("=== Sample Predictions ===\n")
for i in range(min(5, len(res_df))):
    row = res_df.iloc[i]
    print(f"Image: {row['image_id']}")
    print(f"Raw output: {row['prediction_raw'][:500]}")  # First 500 chars
    print(f"Parsed: {row['prediction_vec']}")
    print(f"Expected: {row['human_label']}")
    print("-" * 80)

=== Sample Predictions ===

Image: TE-104.jpg
Raw output: You are classifying a mental-health meme into 7 PHQ-9 symptom indicators.

Indicators in order:
1) Feeling Down
2) Lack of Interest
3) Self-Harm
4) Eating Disorder
5) Low Self-Esteem
6) Concentration Problems
7) Sleeping Disorder

Rules:
- Mark 1 if clearly present, else 0.
- Be conservative; avoid guessing.
- Return exactly this format and nothing else:
PREDICTION: [b1,b2,b3,b4,b5,b6,b7]
- Be conservative; avoid guessing.
Parsed: [0,0,0,0,0,0,0]
Expected: [1,0,0,0,0,0,0]
--------------------------------------------------------------------------------
Image: TE-177.jpg
Raw output: You are classifying a mental-health meme into 7 PHQ-9 symptom indicators.

Indicators in order:
1) Feeling Down
2) Lack of Interest
3) Self-Harm
4) Eating Disorder
5) Low Self-Esteem
6) Concentration Problems
7) Sleeping Disorder

Rules:
- Mark 1 if clearly present, else 0.
- Be conservative; avoid guessing.
- Return exactly this format and nothing els

In [28]:
import re

def _extract_pred(text: str):
    m = re.search(r'PREDICTION:\s*\[([01,\s]+)\]', text, flags=re.IGNORECASE)
    if not m:
        return None
    arr = [x.strip() for x in m.group(1).split(',')]
    if len(arr) == 7 and all(x in ('0','1') for x in arr):
        return '[' + ','.join(arr) + ']'
    return None

final_csv = RESULTS_DIR / 'instructblip_final.csv'
checkpoint_interval = 50

# Start fresh - remove old results
if final_csv.exists():
    final_csv.unlink()

rows = []
print(f"Starting CORRECTED inference on {len(metadata_df)} images...")
print(f"Fix: Decoding only new tokens, not full sequence\n")

for idx, row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):
    iid = row['image_id']
    try:
        image = Image.open(DATA_DIR / row['image_path']).convert('RGB')
        prompt = format_instructblip_prompt()
        inputs = processor(images=image, text=prompt, return_tensors='pt').to(model.device)

        # Generate
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=128)

        # FIX: Decode only NEW tokens (skip input prompt)
        input_length = inputs['input_ids'].shape[1]
        new_tokens = out[:, input_length:]
        text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]

        pred = _extract_pred(text)
        rec = {
            'image_id': iid,
            'category': row['category'],
            'prediction_raw': text,
            'prediction_vec': pred if pred else '[0,0,0,0,0,0,0]',
            'parse_ok': bool(pred),
            'human_label': row['human_label']
        }
    except Exception as e:
        rec = {
            'image_id': iid,
            'category': row['category'],
            'prediction_raw': f'ERROR: {e}',
            'prediction_vec': '[0,0,0,0,0,0,0]',
            'parse_ok': False,
            'human_label': row['human_label']
        }
    rows.append(rec)
    if len(rows) % checkpoint_interval == 0:
        pd.DataFrame(rows).to_csv(final_csv, index=False)
        print(f'✓ Checkpoint: {len(rows)} images')

res_df = pd.DataFrame(rows)
res_df.to_csv(final_csv, index=False)
print(f'\n🎉 Inference complete!')
print(f'✓ Saved: {final_csv}')
print(f'✓ Total images: {len(res_df)}')
print(f'✓ Parse success rate: {res_df["parse_ok"].mean():.1%}')

Starting CORRECTED inference on 650 images...
Fix: Decoding only new tokens, not full sequence



  8%|▊         | 50/650 [00:24<04:59,  2.00it/s]

✓ Checkpoint: 50 images


 15%|█▌        | 100/650 [00:48<04:54,  1.87it/s]

✓ Checkpoint: 100 images


...

✓ Checkpoint: 550 images


 92%|█████████▏| 600/650 [04:42<00:22,  2.19it/s]

✓ Checkpoint: 600 images


100%|██████████| 650/650 [05:06<00:00,  2.12it/s]

✓ Checkpoint: 650 images

🎉 Inference complete!
✓ Saved: /content/drive/MyDrive/colab_experiments/results/instructblip_final.csv
✓ Total images: 650
✓ Parse success rate: 0.0%


In [29]:
# Check what we're getting now
res_df = pd.read_csv(final_csv)

print("=== NEW Sample Predictions ===\n")
for i in range(10):
    row = res_df.iloc[i]
    print(f"Image {i+1}: {row['image_id']}")
    print(f"Raw: '{row['prediction_raw']}'")
    print(f"Length: {len(row['prediction_raw'])} chars")
    print(f"Parsed: {row['prediction_vec']}")
    print(f"Expected: {row['human_label']}")
    print("-" * 80)


=== NEW Sample Predictions ===

Image 1: TE-104.jpg
Raw: '- Be conservative; avoid guessing.'
Length: 34 chars
Parsed: [0,0,0,0,0,0,0]
Expected: [1,0,0,0,0,0,0]
--------------------------------------------------------------------------------
Image 2: TE-177.jpg
Raw: '- Mark 1 if clearly present, else 0.'
Length: 36 chars
Parsed: [0,0,0,0,0,0,0]
Expected: [1,0,0,0,1,0,0]
--------------------------------------------------------------------------------
Image 3: TE-184.jpg
Raw: '- Mark 1 if clearly present, else 0.'
Length: 36 chars
Parsed: [0,0,0,0,0,0,0]
Expected: [1,0,1,0,0,0,0]
--------------------------------------------------------------------------------
Image 4: TE-195.jpg
Raw: '- Mark 1 if clearly present, else 0.'
Length: 36 chars
Parsed: [0,0,0,0,0,0,0]
Expected: [0,0,0,1,1,0,0]
--------------------------------------------------------------------------------
Image 5: TE-285.jpg
Raw: 'No'
Length: 2 chars
Parsed: [0,0,0,0,0,0,0]
Expected: [1,0,1,0,0,0,0]
--------------------------

In [30]:
# Show the actual prompt being used
prompt = format_instructblip_prompt()
print("=== Current Prompt ===")
print(prompt)
print("\n" + "=" * 80)
print(f"Prompt length: {len(prompt)} characters")

=== Current Prompt ===
You are classifying a mental-health meme into 7 PHQ-9 symptom indicators.

Indicators in order:
1) Feeling Down
2) Lack of Interest
3) Self-Harm
4) Eating Disorder
5) Low Self-Esteem
6) Concentration Problems
7) Sleeping Disorder

Rules:
- Mark 1 if clearly present, else 0.
- Be conservative; avoid guessing.
- Return exactly this format and nothing else:
PREDICTION: [b1,b2,b3,b4,b5,b6,b7]


Prompt length: 392 characters


In [31]:
# Create a simpler prompt function
def simple_instructblip_prompt():
    return (
        "Analyze this mental health meme. "
        "Which PHQ-9 symptoms are present? "
        "Respond with exactly: [b1,b2,b3,b4,b5,b6,b7] where each b is 0 or 1. "
        "Symptoms: 1=Feeling Down, 2=Lack of Interest, 3=Self-Harm, 4=Eating Disorder, "
        "5=Low Self-Esteem, 6=Concentration Problems, 7=Sleeping Disorder."
    )

# Test it
test_prompt = simple_instructblip_prompt()
print("=== NEW SIMPLIFIED PROMPT ===")
print(test_prompt)
print(f"\nLength: {len(test_prompt)} chars")

# Quick test on one image
test_row = metadata_df.iloc[0]
test_image = Image.open(DATA_DIR / test_row['image_path']).convert('RGB')
inputs = processor(images=test_image, text=test_prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=100)

input_length = inputs['input_ids'].shape[1]
new_tokens = out[:, input_length:]
response = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]

print("\n=== TEST OUTPUT ===")
print(f"Response: '{response}'")
print(f"Expected format: [1,0,0,0,0,0,0]")

=== NEW SIMPLIFIED PROMPT ===
Analyze this mental health meme. Which PHQ-9 symptoms are present? Respond with exactly: [b1,b2,b3,b4,b5,b6,b7] where each b is 0 or 1. Symptoms: 1=Feeling Down, 2=Lack of Interest, 3=Self-Harm, 4=Eating Disorder, 5=Low Self-Esteem, 6=Concentration Problems, 7=Sleeping Disorder.

Length: 279 chars

=== TEST OUTPUT ===
Response: '0'
Expected format: [1,0,0,0,0,0,0]


In [32]:
# Create summary of Track A (zero-shot baseline) failure
summary = {
    'experiment': 'Track A - InstructBLIP Zero-Shot Baseline',
    'model': 'Salesforce/instructblip-vicuna-7b',
    'images_processed': 650,
    'macro_f1': 0.0,
    'parse_success_rate': 0.0,
    'finding': 'Model unable to follow structured output format in zero-shot setting',
    'evidence': 'Model outputs prompt fragments rather than prediction arrays',
    'implication': 'Establishes need for task-specific prompting (Track B) or fine-tuning (Tracks C/D)',
    'recommendation': 'Proceed with Experiments 2 & 3 using explanation-augmented inputs'
}

import json
with open(RESULTS_DIR / 'track_a_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("=== TRACK A: ZERO-SHOT BASELINE ===\n")
for key, value in summary.items():
    print(f"{key}: {value}")

print("\n✓ Summary saved to: track_a_summary.json")
print("\n📊 CONCLUSION: Zero-shot approach failed. Moving to Track B (PALO-7B with explanations).")

=== TRACK A: ZERO-SHOT BASELINE ===

experiment: Track A - InstructBLIP Zero-Shot Baseline
model: Salesforce/instructblip-vicuna-7b
images_processed: 650
macro_f1: 0.0
parse_success_rate: 0.0
finding: Model unable to follow structured output format in zero-shot setting
evidence: Model outputs prompt fragments rather than prediction arrays
implication: Establishes need for task-specific prompting (Track B) or fine-tuning (Tracks C/D)
recommendation: Proceed with Experiments 2 & 3 using explanation-augmented inputs

✓ Summary saved to: track_a_summary.json

📊 CONCLUSION: Zero-shot approach failed. Moving to Track B (PALO-7B with explanations).


In [33]:
def natural_language_prompt():
    return (
        "Look at this meme carefully. Answer yes or no for each question:\n"
        "1. Does it show feeling sad or depressed?\n"
        "2. Does it show loss of interest in activities?\n"
        "3. Does it mention or hint at self-harm?\n"
        "4. Does it relate to eating problems?\n"
        "5. Does it show low self-esteem or feeling worthless?\n"
        "6. Does it show trouble concentrating?\n"
        "7. Does it relate to sleep problems?\n"
        "Answer in this exact format: 1:yes 2:no 3:no 4:yes 5:no 6:no 7:yes"
    )

# Test on first 3 images
print("=== TESTING NATURAL LANGUAGE PROMPT ===\n")

test_prompt = natural_language_prompt()
print("Prompt:")
print(test_prompt)
print("\n" + "=" * 80 + "\n")

for i in range(3):
    row = metadata_df.iloc[i]
    image = Image.open(DATA_DIR / row['image_path']).convert('RGB')
    inputs = processor(images=image, text=test_prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=150)

    input_length = inputs['input_ids'].shape[1]
    new_tokens = out[:, input_length:]
    response = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]

    print(f"Image {i+1}: {row['image_id']}")
    print(f"Expected: {row['human_label']}")
    print(f"Model response: '{response}'")
    print("-" * 80)

=== TESTING NATURAL LANGUAGE PROMPT ===

Prompt:
Look at this meme carefully. Answer yes or no for each question:
1. Does it show feeling sad or depressed?
2. Does it show loss of interest in activities?
3. Does it mention or hint at self-harm?
4. Does it relate to eating problems?
5. Does it show low self-esteem or feeling worthless?
6. Does it show trouble concentrating?
7. Does it relate to sleep problems?
Answer in this exact format: 1:yes 2:no 3:no 4:yes 5:no 6:no 7:yes


Image 1: TE-104.jpg
Expected: [1,0,0,0,0,0,0]
Model response: ''
--------------------------------------------------------------------------------
Image 2: TE-177.jpg
Expected: [1,0,0,0,1,0,0]
Model response: ''
--------------------------------------------------------------------------------
Image 3: TE-184.jpg
Expected: [1,0,1,0,0,0,0]
Model response: ''
--------------------------------------------------------------------------------


In [34]:
# Deep diagnostic - see raw tokens
test_row = metadata_df.iloc[0]
test_image = Image.open(DATA_DIR / test_row['image_path']).convert('RGB')
test_prompt = natural_language_prompt()

inputs = processor(images=test_image, text=test_prompt, return_tensors='pt').to(model.device)

print("=== TOKEN GENERATION DIAGNOSTIC ===\n")
print(f"Input prompt length: {inputs['input_ids'].shape[1]} tokens")

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        num_beams=1,
        return_dict_in_generate=True,
        output_scores=True
    )

generated_ids = out.sequences[0]
print(f"Total generated length: {len(generated_ids)} tokens")
print(f"New tokens generated: {len(generated_ids) - inputs['input_ids'].shape[1]}")

# Decode full sequence
full_text = processor.decode(generated_ids, skip_special_tokens=True)
print(f"\nFull decoded text ({len(full_text)} chars):")
print(f"'{full_text}'")

# Decode just new tokens
input_length = inputs['input_ids'].shape[1]
new_token_ids = generated_ids[input_length:]
new_text = processor.decode(new_token_ids, skip_special_tokens=True)
print(f"\nNew tokens only ({len(new_text)} chars):")
print(f"'{new_text}'")

# Show raw token IDs for new tokens
print(f"\nRaw new token IDs: {new_token_ids.tolist()[:20]}")  # First 20

=== TOKEN GENERATION DIAGNOSTIC ===

Input prompt length: 167 tokens
Total generated length: 169 tokens
New tokens generated: 2

Full decoded text (430 chars):
'Look at this meme carefully. Answer yes or no for each question:
1. Does it show feeling sad or depressed?
2. Does it show loss of interest in activities?
3. Does it mention or hint at self-harm?
4. Does it relate to eating problems?
5. Does it show low self-esteem or feeling worthless?
6. Does it show trouble concentrating?
7. Does it relate to sleep problems?
Answer in this exact format: 1:yes 2:no 3:no 4:yes 5:no 6:no 7:yes'

New tokens only (0 chars):
''

Raw new token IDs: [2, 1]


In [36]:
def list_symptoms_prompt():
    return (
        "List all mental health issues shown in this meme. "
        "Options: depression, loss of interest, self-harm, eating problems, "
        "low self-esteem, concentration issues, sleep problems. "
        "List all that apply, separated by commas."
    )

# Test on 5 different images
print("=== TESTING LIST-BASED PROMPT ===\n")

test_prompt = list_symptoms_prompt()
print(f"Prompt: '{test_prompt}'")
print("\n" + "=" * 80 + "\n")

# Test on images with different label patterns
test_indices = [0, 1, 2, 50, 100]  # Mix of different images

for idx in test_indices:
    row = metadata_df.iloc[idx]
    image = Image.open(DATA_DIR / row['image_path']).convert('RGB')
    inputs = processor(images=image, text=test_prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    input_length = inputs['input_ids'].shape[1]
    new_tokens = out[:, input_length:]
    response = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]

    print(f"Image: {row['image_id']}")
    print(f"Expected: {row['human_label']}")
    print(f"Response: '{response}'")
    print("-" * 80)

=== TESTING LIST-BASED PROMPT ===

Prompt: 'List all mental health issues shown in this meme. Options: depression, loss of interest, self-harm, eating problems, low self-esteem, concentration issues, sleep problems. List all that apply, separated by commas.'


Image: TE-104.jpg
Expected: [1,0,0,0,0,0,0]
Response: 'depression, loss of interest, self-harm, eating problems, low self-esteem, concentration issues, sleep problems'
--------------------------------------------------------------------------------
Image: TE-177.jpg
Expected: [1,0,0,0,1,0,0]
Response: 'depression, loss of interest'
--------------------------------------------------------------------------------
Image: TE-184.jpg
Expected: [1,0,1,0,0,0,0]
Response: 'depression'
--------------------------------------------------------------------------------
Image: TE-306.jpg
Expected: [0,0,0,1,0,0,0]
Response: 'depression, loss of interest, self-harm, eating problems, low self-esteem, concentration issues, sleep problems'
--------

In [35]:
def open_ended_prompt():
    return (
        "What mental health issues does this meme express? "
        "Consider: depression, loss of interest, self-harm, eating problems, "
        "low self-esteem, concentration issues, sleep problems."
    )

# Test on 3 images
print("=== TESTING OPEN-ENDED PROMPT ===\n")

test_prompt = open_ended_prompt()
print(f"Prompt: '{test_prompt}'")
print("\n" + "=" * 80 + "\n")

for i in range(3):
    row = metadata_df.iloc[i]
    image = Image.open(DATA_DIR / row['image_path']).convert('RGB')
    inputs = processor(images=test_image, text=test_prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    input_length = inputs['input_ids'].shape[1]
    new_tokens = out[:, input_length:]
    response = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]

    print(f"Image {i+1}: {row['image_id']}")
    print(f"Expected: {row['human_label']}")
    print(f"Response ({len(response)} chars): '{response}'")
    print("-" * 80)

=== TESTING OPEN-ENDED PROMPT ===

Prompt: 'What mental health issues does this meme express? Consider: depression, loss of interest, self-harm, eating problems, low self-esteem, concentration issues, sleep problems.'


Image 1: TE-104.jpg
Expected: [1,0,0,0,0,0,0]
Response (10 chars): 'depression'
--------------------------------------------------------------------------------
Image 2: TE-177.jpg
Expected: [1,0,0,0,1,0,0]
Response (10 chars): 'depression'
--------------------------------------------------------------------------------
Image 3: TE-184.jpg
Expected: [1,0,1,0,0,0,0]
Response (10 chars): 'depression'
--------------------------------------------------------------------------------


In [37]:
def list_symptoms_prompt():
    return (
        "List all mental health issues shown in this meme. "
        "Options: depression, loss of interest, self-harm, eating problems, "
        "low self-esteem, concentration issues, sleep problems. "
        "List all that apply, separated by commas."
    )

def parse_symptom_list(response_text):
    """Convert natural language response to binary vector [1,0,1,...]"""
    response_lower = response_text.lower()

    # Map symptoms to positions
    symptom_map = {
        'depression': 0,
        'loss of interest': 1,
        'self-harm': 2,
        'eating problems': 3,
        'low self-esteem': 4,
        'concentration issues': 5,
        'sleep problems': 6
    }

    vec = [0] * 7
    for symptom, idx in symptom_map.items():
        if symptom in response_lower:
            vec[idx] = 1

    return '[' + ','.join(str(x) for x in vec) + ']'

# Run full inference
final_csv = RESULTS_DIR / 'instructblip_natural_language.csv'

print("=== FULL NATURAL LANGUAGE INFERENCE ===")
print(f"Prompt: {list_symptoms_prompt()}")
print(f"\nProcessing {len(metadata_df)} images...")
print("This will take ~5-10 minutes\n")

rows = []
for idx, row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):
    try:
        image = Image.open(DATA_DIR / row['image_path']).convert('RGB')
        prompt = list_symptoms_prompt()
        inputs = processor(images=image, text=prompt, return_tensors='pt').to(model.device)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=100,
                do_sample=True,
                temperature=0.7,
                top_p=0.9
            )

        input_length = inputs['input_ids'].shape[1]
        new_tokens = out[:, input_length:]
        response = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]

        pred_vec = parse_symptom_list(response)

        rec = {
            'image_id': row['image_id'],
            'category': row['category'],
            'prediction_raw': response,
            'prediction_vec': pred_vec,
            'human_label': row['human_label']
        }
    except Exception as e:
        rec = {
            'image_id': row['image_id'],
            'category': row['category'],
            'prediction_raw': f'ERROR: {e}',
            'prediction_vec': '[0,0,0,0,0,0,0]',
            'human_label': row['human_label']
        }
    rows.append(rec)

    if len(rows) % 50 == 0:
        pd.DataFrame(rows).to_csv(final_csv, index=False)
        print(f'✓ Checkpoint: {len(rows)} images')

res_df = pd.DataFrame(rows)
res_df.to_csv(final_csv, index=False)

print(f'\n🎉 Inference complete!')
print(f'✓ Saved: {final_csv}')
print(f'✓ Total images: {len(res_df)}')

=== FULL NATURAL LANGUAGE INFERENCE ===
Prompt: List all mental health issues shown in this meme. Options: depression, loss of interest, self-harm, eating problems, low self-esteem, concentration issues, sleep problems. List all that apply, separated by commas.

Processing 650 images...
This will take ~5-10 minutes



  8%|▊         | 50/650 [00:38<07:27,  1.34it/s]

✓ Checkpoint: 50 images


 15%|█▌        | 100/650 [01:16<09:19,  1.02s/it]

✓ Checkpoint: 100 images


...

✓ Checkpoint: 550 images


 92%|█████████▏| 600/650 [06:29<00:21,  2.35it/s]

✓ Checkpoint: 600 images


100%|██████████| 650/650 [07:02<00:00,  1.54it/s]

✓ Checkpoint: 650 images

🎉 Inference complete!
✓ Saved: /content/drive/MyDrive/colab_experiments/results/instructblip_natural_language.csv
✓ Total images: 650


In [38]:
# Load results and calculate metrics
res_df = pd.read_csv(RESULTS_DIR / 'instructblip_natural_language.csv')

# Parse predictions
y_true = parse_predictions(res_df['human_label'].tolist())
y_pred = parse_predictions(res_df['prediction_vec'].tolist())

# Calculate metrics
metrics = calculate_metrics(y_true, y_pred, SYMPTOM_NAMES)

print("\n" + "=" * 80)
print("=== INSTRUCTBLIP NATURAL LANGUAGE RESULTS ===")
print("=" * 80)
print_results_table(metrics)

# Save metrics
save_metrics_csv(metrics, RESULTS_DIR / 'instructblip_natural_language_metrics.csv')

import json
with open(RESULTS_DIR / 'instructblip_natural_language_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("\n✓ Metrics saved!")

# Show some sample predictions
print("\n" + "=" * 80)
print("=== SAMPLE PREDICTIONS ===")
print("=" * 80)
for i in range(5):
    row = res_df.iloc[i]
    print(f"\nImage: {row['image_id']}")
    print(f"Model said: '{row['prediction_raw']}'")
    print(f"Predicted: {row['prediction_vec']}")
    print(f"Actual:    {row['human_label']}")



=== INSTRUCTBLIP NATURAL LANGUAGE RESULTS ===
n=650
Macro-F1=0.2500 | Weighted-F1=0.3021 | Macro-P=0.1621 | Macro-R=0.5607
--------------------------------------------------------------------------------
Symptom                              P       R      F1    Gold+    Pred+
--------------------------------------------------------------------------------
Feeling Down                     0.341   0.944   0.501      215      596
Lack of Interest                 0.092   0.431   0.151       65      306
Self-Harm                        0.135   0.532   0.215       79      312
Eating Disorder                  0.163   0.598   0.257       82      300
Low Self-Esteem                  0.178   0.514   0.265      111      320
Concentration Problems           0.108   0.458   0.175       72      305
Sleeping Disorder                0.119   0.449   0.188       78      295

✓ Metrics saved!

=== SAMPLE PREDICTIONS ===

Image: TE-104.jpg
Model said: 'Depression, loss of interest, self-harm, eating prob

In [39]:
import os

print("=== CHECKING SAVED FILES ===\n")

results_dir = RESULTS_DIR
files_to_check = [
    'instructblip_natural_language.csv',
    'instructblip_natural_language_metrics.csv',
    'instructblip_natural_language_metrics.json'
]

print(f"Results directory: {results_dir}\n")

for filename in files_to_check:
    filepath = results_dir / filename
    exists = filepath.exists()
    size = filepath.stat().st_size / 1024 if exists else 0  # KB

    status = "✅ SAVED" if exists else "❌ MISSING"
    print(f"{status} - {filename} ({size:.1f} KB)")

# List all files in results/
print(f"\n=== ALL FILES IN results/ ===")
for item in sorted(os.listdir(results_dir)):
    filepath = results_dir / item
    size = filepath.stat().st_size / 1024
    print(f"  {item} ({size:.1f} KB)")

print("\n✅ All files are in Google Drive and will persist after session ends")

=== CHECKING SAVED FILES ===

Results directory: /content/drive/MyDrive/colab_experiments/results

✅ SAVED - instructblip_natural_language.csv (76.0 KB)
✅ SAVED - instructblip_natural_language_metrics.csv (0.6 KB)
✅ SAVED - instructblip_natural_language_metrics.json (1.6 KB)

=== ALL FILES IN results/ ===
  instructblip_final.csv (64.1 KB)
  instructblip_natural_language.csv (76.0 KB)
  instructblip_natural_language_metrics.csv (0.6 KB)
  instructblip_natural_language_metrics.json (1.6 KB)
  track_a_summary.json (0.5 KB)

✅ All files are in Google Drive and will persist after session ends


In [40]:
from google.colab import files

print("Downloading results to your computer...\n")

# Download all result files
files.download(str(RESULTS_DIR / 'instructblip_natural_language.csv'))
files.download(str(RESULTS_DIR / 'instructblip_natural_language_metrics.csv'))
files.download(str(RESULTS_DIR / 'instructblip_natural_language_metrics.json'))

print("✅ Downloads complete! Check your browser's download folder.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloads complete! Check your browser's download folder.


In [41]:
# Create comprehensive summary
summary = f"""
# EXPERIMENT 1 COMPLETE - InstructBLIP Zero-Shot Baseline

## Date: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}

## Model Configuration
- Model: Salesforce/instructblip-vicuna-7b
- GPU: NVIDIA A100-SXM4-40GB (Google Colab Pro)
- Approach: Zero-shot with natural language prompting

## Dataset
- Total images: 650 Hindi mental health memes
- Test set from: translated_categorized_memes/*/test/

## Prompt Used
"{list_symptoms_prompt()}"

## Results Summary
- **Macro-F1: 0.2500 (25%)**
- **Weighted-F1: 0.3021 (30%)**
- **Macro Precision: 0.1621 (16%)**
- **Macro Recall: 0.5607 (56%)**

## Per-Symptom Performance
- Feeling Down: 50.1% F1 (BEST)
- Lack of Interest: 15.1% F1
- Self-Harm: 21.5% F1
- Eating Disorder: 25.7% F1
- Low Self-Esteem: 26.5% F1
- Concentration Problems: 17.5% F1
- Sleeping Disorder: 18.8% F1

## Key Findings
1. ✅ Natural language prompting works (25% vs 0% for structured output)
2. ⚠️ Model exhibits over-prediction bias (high recall, low precision)
3. ✅ "Feeling Down" easiest symptom to detect (50% F1)
4. ✅ Establishes baseline for Track B/C/D experiments

## Files Saved
- instructblip_natural_language.csv (predictions for 650 images)
- instructblip_natural_language_metrics.csv (F1 scores)
- instructblip_natural_language_metrics.json (full metrics)

## Runtime
- Total inference time: ~7 minutes
- Speed: ~1.5 images/second on A100

## Next Steps
- Experiment 2: PALO-7B with explanation + image input
- Experiment 3: BLIP-2 with explanation + image input
- Expected improvement: 35-50% macro-F1 with explanations
"""

# Save summary
summary_path = RESULTS_DIR / 'experiment_1_summary.txt'
with open(summary_path, 'w') as f:
    f.write(summary)

print(summary)
print(f"\n✅ Summary saved to: {summary_path}")

# Also download it
files.download(str(summary_path))


# EXPERIMENT 1 COMPLETE - InstructBLIP Zero-Shot Baseline

## Date: 2026-04-01 04:59

## Model Configuration
- Model: Salesforce/instructblip-vicuna-7b
- GPU: NVIDIA A100-SXM4-40GB (Google Colab Pro)
- Approach: Zero-shot with natural language prompting

## Dataset
- Total images: 650 Hindi mental health memes
- Test set from: translated_categorized_memes/*/test/

## Prompt Used
"List all mental health issues shown in this meme. Options: depression, loss of interest, self-harm, eating problems, low self-esteem, concentration issues, sleep problems. List all that apply, separated by commas."

## Results Summary
- **Macro-F1: 0.2500 (25%)**
- **Weighted-F1: 0.3021 (30%)**
- **Macro Precision: 0.1621 (16%)**
- **Macro Recall: 0.5607 (56%)**

## Per-Symptom Performance
- Feeling Down: 50.1% F1 (BEST)
- Lack of Interest: 15.1% F1
- Self-Harm: 21.5% F1
- Eating Disorder: 25.7% F1
- Low Self-Esteem: 26.5% F1
- Concentration Problems: 17.5% F1
- Sleeping Disorder: 18.8% F1

## Key Findings
1.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>